PROJET MODELISATION GEOMETRIQUE

MODELISATION D'UNe INVASION ZOMBIE

Partie 1 : Mouvement Aléatoire

In [ ]:
#IMPORTS 

import numpy as np
from scipy.spatial import KDTree
import plotly.graph_objects as go
import random


In [ ]:
# Paramètres généraux
nb_explorateurs = 80 
nb_rapides = 8
zone_limite = 60
precision = 1000
infectes_initiaux = 1
aoki = False
chasse = False
fuite = False
cone = False
frames_data = []
dt = 0.1  
nb_frames = 400


# Paramètres de Aoki et Infection
rayons = {'rep': 10.0, 'align': 20.0, 'att': 40.0, 'infect': 15.0, 'zombie_rep': 50.0, 'zombie_chasse': 70.0}
parametres_k = {'rep': 2.0, 'align': 1.0, 'att': 0.5, 'zombie_rep': 5.0, 'zombie_chasse': 8.0, 'v_max_sain': 12.0, 'v_max_zombie': 25.0, 'v_max_rapide': 45.0}

In [ ]:


#Les explorateurs sont initialisés avec des positions et des vitesses aléatoires dans un espace de taille définie par zone_limite. Les vitesses sont également aléatoires, ce qui permet de simuler un mouvement naturel des explorateurs dans le vaisseau (sans gravité).
#Chaque explorateur est représenté par un identifiant unique (i), une position (x, y, z) et une vitesse (vx, vy, vz). Le paramètre False à la fin de la ligne d'ajout de l'explorateur peut être utilisé pour indiquer si l'explorateur a été contaminé par un zombie ou non.
def initialiser_explorateurs(nb_explorateurs,zone_limite,precision):
    explorateurs = []
    for i in range(nb_explorateurs):
        # On centre les particules entre -zone_limite et +zone_limite
        x = random.randint(-precision, precision) * zone_limite / precision
        y = random.randint(-precision, precision) * zone_limite / precision
        z = random.randint(-precision, precision) * zone_limite / precision
        vx = random.randint(-precision, precision) * 10 / precision
        vy = random.randint(-precision, precision) * 10 / precision
        vz = random.randint(-precision, precision) * 10 / precision
        # [ID, Position_Array, Vitesse_Array, Est_Infecté, Est_Rapide]
        est_rapide = True if i < nb_rapides else False
        explorateurs.append([i, np.array([x, y, z]), np.array([vx, vy, vz]), False, est_rapide])
    return explorateurs

def contaminer_explorateur(explorateurs,infectes_initiaux):
    for _ in range(infectes_initiaux):
        index = random.randint(0, len(explorateurs) - 1)
        while (explorateurs[index][3] == True):
            index = random.randint(0, len(explorateurs) - 1)
        explorateurs[index][3] = True





In [ ]:

def calculer_forces_completes(explorateurs, aoki, chasse, fuite, cone):
    N = len(explorateurs)
    pos_all = np.array([e[1] for e in explorateurs])
    vit_all = np.array([e[2] for e in explorateurs])
    etats_all = np.array([e[3] for e in explorateurs])
    rapides_all = np.array([e[4] for e in explorateurs])
    
    arbre = KDTree(pos_all)
    rayon_max = max(rayons.values())
    nouvelles_vitesses = []

    for i in range(N):
        p_i, v_i, etat_i, est_rapide_i = pos_all[i], vit_all[i], etats_all[i], rapides_all[i]
        F_rep, F_align, F_att, F_fuite, F_chasse = np.zeros(3), np.zeros(3), np.zeros(3), np.zeros(3), np.zeros(3)
        n_align = 0
        
        voisins_idx = arbre.query_ball_point(p_i, rayon_max)
        for j in voisins_idx:
            if i == j: continue
            p_j, v_j, etat_j = pos_all[j], vit_all[j], etats_all[j]
            d_vec = p_j - p_i
            dist = np.linalg.norm(d_vec)
            if dist == 0: continue
            u = d_vec / dist
            
            #Cone de vision
            if cone : 
                v_norm_i = np.linalg.norm(v_i)
                if v_norm_i > 0:
                    v_i_dir = v_i / v_norm_i
                    cos_angle = np.dot(v_i_dir, u)
                    if cos_angle < 0.5: 
                        continue 

            # Infection
            if dist < rayons['infect'] and (etat_i != etat_j):
                explorateurs[i][3] = explorateurs[j][3] = True
                etat_i = True

            # Fuite/Chasse
            if fuite :
                if not etat_i and etat_j: 
                    if dist < rayons['zombie_rep']: F_fuite += -parametres_k['zombie_rep'] * u
            if chasse :    
                if etat_i and not etat_j: 
                    if dist < rayons['zombie_chasse']: F_chasse += parametres_k['zombie_chasse'] * u

            # Aoki
            if aoki : 
                if etat_i == etat_j:
                    if dist < rayons['rep']: F_rep += -parametres_k['rep'] * u
                    elif dist < rayons['align']: 
                        F_align += v_j
                        n_align += 1
                    elif dist < rayons['att']: F_att += parametres_k['att'] * u
                    
        if n_align > 0: F_align = (F_align/n_align) * parametres_k['align']
        
        force_totale = F_rep + F_align + F_att + F_fuite + F_chasse
        
        # L'accélérateur des éclaireurs
        if est_rapide_i and not etat_i: 
            force_totale *= 4.0
            
        # Sélection de la limite de vitesse
        if etat_i: 
            v_limit = parametres_k['v_max_zombie']
        elif est_rapide_i: 
            v_limit = parametres_k['v_max_rapide']
        else: 
            v_limit = parametres_k['v_max_sain']
            
        nv_vit = v_i + force_totale
        norme = np.linalg.norm(nv_vit)
        if norme > v_limit: nv_vit = (nv_vit / norme) * v_limit
        nouvelles_vitesses.append(nv_vit)
        
    return nouvelles_vitesses

In [ ]:
def update_explorateurs(explorateurs, dt, aoki, chasse, fuite, cone):
    #On selectionne la fonction de calcul des forces en fonction du choix du cone de vision. 
    vits = calculer_forces_completes(explorateurs, aoki, chasse, fuite, cone)
    for i in range(len(explorateurs)):
        explorateurs[i][1] += vits[i] * dt
        explorateurs[i][2] = vits[i]
        
        # Murs
        for axe in range(3):
            if explorateurs[i][1][axe] > zone_limite:
                explorateurs[i][1][axe] = zone_limite
                explorateurs[i][2][axe] *= -1
            elif explorateurs[i][1][axe] < -zone_limite:
                explorateurs[i][1][axe] = -zone_limite
                explorateurs[i][2][axe] *= -1

In [ ]:


#On génère les positions et vitesses initiales des explorateurs. 
explorateurs = initialiser_explorateurs(nb_explorateurs,zone_limite,precision)
#On infecte un explorateur
contaminer_explorateur(explorateurs,infectes_initiaux)

for frame in range(nb_frames):
    x = [e[1][0] for e in explorateurs]
    y = [e[1][1] for e in explorateurs]
    z = [e[1][2] for e in explorateurs]
    infect = [e[3] for e in explorateurs]
    rapides = [e[4] for e in explorateurs]
    frames_data.append((x, y, z, infect, rapides))
    update_explorateurs(explorateurs, dt, aoki, chasse, fuite, cone)

x_cage = [-1, 1, 1, -1, -1, -1, 1, 1, -1, -1, 1, 1, 1, 1, -1, -1]
y_cage = [-1, -1, 1, 1, -1, -1, -1, 1, 1, -1, -1, -1, 1, 1, 1, 1]
z_cage = [-1, -1, -1, -1, -1, 1, 1, 1, 1, 1, 1, -1, -1, 1, 1, -1]

x_cage = [x * zone_limite for x in x_cage]
y_cage = [y * zone_limite for y in y_cage]
z_cage = [z * zone_limite for z in z_cage]

couleurs_initiales = ['red' if inf else ('gold' if rap else 'blue') for inf, rap in zip(frames_data[0][3], frames_data[0][4])]
symbol_initiaux=['cross' if est_infecte else 'circle' for est_infecte in frames_data[0][3]]

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=x_cage, y=y_cage, z=z_cage, 
            mode='lines', 
            line=dict(color='cyan', width=4)),

        go.Scatter3d(
            x=frames_data[0][0], 
            y=frames_data[0][1], 
            z=frames_data[0][2], 
            mode='markers', 
            marker=dict(size=5, color=couleurs_initiales, symbol=symbol_initiaux))]
)

# Pour chaque frame, on met à jour les positions et les couleurs
frames_tempo = []

for frame in frames_data:
    couleurs = ['red' if inf else ('gold' if rap else 'blue') for inf, rap in zip(frame[3], frame[4])]
    symbol = ['cross' if est_infecte else 'circle' for est_infecte in frame[3]]
    frames_tempo.append(
    go.Frame(data=[go.Scatter3d(x=x_cage, y=y_cage, z=z_cage, mode='lines', line=dict(color='cyan', width=4)),
                    go.Scatter3d(x=frame[0], y=frame[1], z=frame[2], marker=dict(size=5, color=couleurs, symbol=symbol))
                   ])
)

fig.frames = frames_tempo

no_axis = dict(
    showbackground=False,
    showticklabels=False,
    showgrid=False,
    showline=False,
    zeroline=False
)

fig.update_layout(
    title='Simulation des explorateurs flottants dans le vaisseau',
    paper_bgcolor='black',
    scene=dict(
        xaxis=dict(**no_axis, range=[-zone_limite, zone_limite]),
        yaxis=dict(**no_axis, range=[-zone_limite, zone_limite]),
        zaxis=dict(**no_axis, range=[-zone_limite, zone_limite]),
        aspectmode='cube'
    ),
    updatemenus=[dict(
        type='buttons',
        buttons=[dict(label='Play', method='animate', args=[None, {'frame': {'duration': 50, 'redraw': True}, 'fromcurrent': True}])]
    )]
)

fig.show()